In [2]:
import osmnx as ox
import os
import pandas as pd
cache_dir = os.path.join(os.path.expanduser("~"), "osmnx_cache")
ox.config(cache_folder=cache_dir)
df = pd.read_csv("D:/cbva/backend/intersecciones.csv", sep=';',encoding='utf-8')
import ast
df['Intersecciones'] = df['Intersecciones'].apply(ast.literal_eval)
df

C:\Users\savil\AppData\Local\Temp\ipykernel_16868\2800829195.py:5: FutureWarning: The `utils.config` function is deprecated and will be removed in the v2.0.0 release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  ox.config(cache_folder=cache_dir)


,Calles,Intersecciones


In [3]:
def get_all_street_names(city):
    # Load the street network for the specified city
    G = ox.graph_from_place(city, network_type='drive')

    # Convert the network to a GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False)

    # Extract street names, handling cases where the name is a list
    street_names = set()
    for name in edges['name']:
        if isinstance(name, list):
            street_names.update(name)
        else:
            street_names.add(name)

    # Remove None values if present
    street_names.discard(None)

    return street_names
def get_intersections(city, street_name):
    # Load the street network for the specified city
    G = ox.graph_from_place(city, network_type='drive')

    # Convert the network to a GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False)

    # Filter edges that contain the specified street
    filtered_edges = edges[edges['name'].apply(lambda x: street_name in x if isinstance(x, list) else street_name == x)]

    # Find nodes connected to the filtered edges
    nodes_connected = set()
    for _, edge in filtered_edges.iterrows():
        # Get nodes from the geometry of the edge
        for point in list(edge['geometry'].coords):
            nodes = ox.distance.nearest_nodes(G, point[0], point[1])
            nodes_connected.add(nodes)

    # For each node, find intersecting streets
    intersecting_streets = set()
    for node in nodes_connected:
        # Get edges connected to this node
        edges_connected = G.edges(node, data=True)
        for edge in edges_connected:
            street_data = edge[2].get('name', None)
            if street_data:
                # Handle cases where street name is a list
                if isinstance(street_data, list):
                    for street in street_data:
                        intersecting_streets.add(street)
                else:
                    intersecting_streets.add(street_data)

    return list(intersecting_streets)
def get_intersection_coordinates(G, street1, street2):
    # Load the street network for the specified city

    # Find all nodes that are part of each street
    nodes_street1 = set()
    nodes_street2 = set()

    # Iterate over all edges in the graph
    for u, v, data in G.edges(data=True):
        if 'name' in data:
            # Check if this edge is part of street1
            if isinstance(data['name'], list):
                if street1 in data['name']:
                    nodes_street1.update([u, v])
            else:
                if street1 == data['name']:
                    nodes_street1.update([u, v])

            # Check if this edge is part of street2
            if isinstance(data['name'], list):
                if street2 in data['name']:
                    nodes_street2.update([u, v])
            else:
                if street2 == data['name']:
                    nodes_street2.update([u, v])

    # Find intersection node(s) - nodes that are part of both streets
    intersection_nodes = nodes_street1.intersection(nodes_street2)

    # Get coordinates for the intersection node(s)
    intersection_coords = []
    for node in intersection_nodes:
        x = G.nodes[node]['x']
        y = G.nodes[node]['y']
        intersection_coords.append((y, x))

    return intersection_coords

In [4]:
city = "Villa Alemana, Chile"
G = ox.graph_from_place(city, network_type='drive')
calles = get_all_street_names(city)
calles

{'Sevillana',
 'Pasaje Granizo',
 'Pasaje Las Cruces',
 'Bilbao',
 'Pasaje Eugenio Grandi',
 'Pasaje Sandy Zañartu',
 'Rafael Fabres',
 'Pasaje Prat',
 'Prieto',
 'Lastra',
 'Avenida Las Américas',
 'Cardenal José María Caro',
 'Pasaje La Lenga',
 'Camilo Mori',
 'Fernanda',
 'Pasaje Fragata Leonora',
 'Fernandito',
 'Claudio Gay',
 'Pasaje Alfonso Alcalde',
 'Rodríguez',
 'Pasaje Guacolda',
 'Pasaje Cinabrio',
 'Jesuitas',
 'Pasaje Gran Canaria',
 'Pasaje Los Mañíos',
 'Presbítero Osvaldo Lira Pérez',
 'Haydn',
 'San Rafael',
 'San Gerónimo',
 'Pasaje Tres Sur',
 'Pasaje Chile',
 'Cerro Alto Pucará',
 'Magisterio Poniente',
 nan,
 'Olmué',
 'Beethoven',
 'Raúl González',
 'Pasaje Carlos Ibáñez',
 'Tucapel',
 'Togo',
 'Pasaje 10',
 'La Boira',
 'Amunátegui',
 'Jardines de Aranda III',
 'Pasaje Gabriel Dazarola',
 'Pasaje Perseo',
 'Carcaroly',
 'Los Maitenes',
 'Las Piedras',
 'Pasaje Latorre',
 'Gandarillas',
 'Pasaje Haydn',
 'Lima',
 'Pasaje Río Valdivia 3',
 'Maranata',
 'Pasaje La

In [7]:
calles = list(calles)
calles

['Sevillana',
 'Pasaje Granizo',
 'Pasaje Las Cruces',
 'Bilbao',
 'Pasaje Eugenio Grandi',
 'Pasaje Sandy Zañartu',
 'Rafael Fabres',
 'Pasaje Prat',
 'Prieto',
 'Lastra',
 'Avenida Las Américas',
 'Cardenal José María Caro',
 'Pasaje La Lenga',
 'Camilo Mori',
 'Fernanda',
 'Pasaje Fragata Leonora',
 'Fernandito',
 'Claudio Gay',
 'Pasaje Alfonso Alcalde',
 'Rodríguez',
 'Pasaje Guacolda',
 'Pasaje Cinabrio',
 'Jesuitas',
 'Pasaje Gran Canaria',
 'Pasaje Los Mañíos',
 'Presbítero Osvaldo Lira Pérez',
 'Haydn',
 'San Rafael',
 'San Gerónimo',
 'Pasaje Tres Sur',
 'Pasaje Chile',
 'Cerro Alto Pucará',
 'Magisterio Poniente',
 nan,
 'Olmué',
 'Beethoven',
 'Raúl González',
 'Pasaje Carlos Ibáñez',
 'Tucapel',
 'Togo',
 'Pasaje 10',
 'La Boira',
 'Amunátegui',
 'Jardines de Aranda III',
 'Pasaje Gabriel Dazarola',
 'Pasaje Perseo',
 'Carcaroly',
 'Los Maitenes',
 'Las Piedras',
 'Pasaje Latorre',
 'Gandarillas',
 'Pasaje Haydn',
 'Lima',
 'Pasaje Río Valdivia 3',
 'Maranata',
 'Pasaje La

In [11]:
df_temp = pd.read_csv("D:/cbva/backend/intersecciones.csv", sep=';',encoding='utf-8')
calles_temp = df_temp['Calles'].tolist()
for i in calles_temp:
    if i in calles:
        calles.remove(i)
calles

[nan,
 'Olmué',
 'Beethoven',
 'Raúl González',
 'Pasaje Carlos Ibáñez',
 'Tucapel',
 'Togo',
 'Pasaje 10',
 'La Boira',
 'Amunátegui',
 'Jardines de Aranda III',
 'Pasaje Gabriel Dazarola',
 'Pasaje Perseo',
 'Carcaroly',
 'Los Maitenes',
 'Las Piedras',
 'Pasaje Latorre',
 'Gandarillas',
 'Pasaje Haydn',
 'Lima',
 'Pasaje Río Valdivia 3',
 'Maranata',
 'Pasaje Las Dalias',
 'Taitao',
 'Pasaje Tacna',
 'Calle Villa Esmeralda',
 'Pasaje Progreso',
 'Pasaje San Pablo',
 'Mare',
 'Los Dominicos',
 'Calle Del Llano',
 'Colmar',
 'Suecia',
 'Los Crespones',
 'Pasaje Seis',
 'Pasaje Rubi',
 'Dalias',
 'Pasaje 5',
 'El Espino',
 'Pasaje Los Huemules De Niblinto',
 'Presbítero Luis Fernández Carnero',
 'Pasaje Lircay',
 'Pasaje Santa Macarena',
 'Pasaje Isla Negra',
 'El Arreo',
 'Capitán Ávalos',
 'Pasaje Las Lilas',
 'Pasaje Lago Calafquén',
 'Pasaje La Retuca',
 'Pasaje Gonzalo Rojas',
 'Sergio Aguirre',
 'Proa',
 'Pasaje Alba De Tormes',
 'Pasaje Williamson',
 'Condominio San Enrique',
 '

In [12]:
for calle in calles:
    street_name = calle
    index = calles.index(calle)
    print(f'Calle {index+1} de {len(calles)}: {street_name}')
    intersections = get_intersections(city, street_name)
    if street_name in intersections:
        intersections.remove(street_name)
    print(intersections)
    street_data = [street_name, intersections]
    df.loc[len(df.index)] = street_data
    df.to_csv('./intersecciones.csv', sep=';', index=False,encoding='utf-8')

Calle 1 de 1227: nan
[]
Calle 2 de 1227: Olmué
['Limache', 'Pasaje La Retuca', 'Lliu Lliu', 'Pasaje Los Yuyos', 'Carlos Saavedra Oriente']
Calle 3 de 1227: Beethoven
['Pasaje Wagner', 'Ojos de Agua', 'Mozart', 'Haydn', 'Lajos Janosa', 'Pasaje Schubert']
Calle 4 de 1227: Raúl González
['Canadá', 'Los Mejillones', 'Pasaje Calderilla', 'Pasaje Los Hornitos', 'Pasaje Caldera', 'Pasaje Puerto Coloso', 'Punta Arenas', 'Portofino', 'Pasaje Esmeralda', 'Pasaje Tocopilla', 'Madrid']
Calle 5 de 1227: Pasaje Carlos Ibáñez
['Pasaje Santiago', 'Bogotá', 'General Carlos Ibáñez del Campo']
Calle 6 de 1227: Tucapel
['Llacolen', 'Julio Cepeda', 'San Jorge', 'Pasaje Tassara', 'Santa Elena', 'Gustavo Zamora', 'Lautaro', 'Colo Colo', 'Parroquia', 'Pasaje 2', 'San Carlos']
Calle 7 de 1227: Togo
['Tokio', 'Domingo Composto', 'Sargento Aldea', 'Madrid']
Calle 8 de 1227: Pasaje 10
['Alcalde Alejandro Peralta', 'Los Abedules']
Calle 9 de 1227: La Boira
['La Niebla']
Calle 10 de 1227: Amunátegui
['Baquedano', '

In [13]:
calles = df['Calles'].tolist()
inters = df['Intersecciones'].tolist()
inters2 = []
X = []
Y = []
for i in calles:
    index = calles.index(i)
    int = inters[index]
    inters_inside = []
    X_inside = []
    Y_inside = []
    for j in int:
        print(f'{i} con {j}')
        intersection_coords = get_intersection_coordinates(G, i, j)
        print(intersection_coords)
        if len(intersection_coords)>0:
            inters_inside.append(j)
            X_inside.append(intersection_coords[0][0])
            Y_inside.append(intersection_coords[0][1])
    inters2.append(inters_inside)
    X.append(X_inside)
    Y.append(Y_inside)
print(X)
print(Y)

Sevillana con Porvenir
[(-33.0479902, -71.3927166)]
Pasaje Granizo con Las Piedras
[(-33.06245, -71.3835087)]
Pasaje Granizo con Jarilla
[(-33.0624713, -71.3823881)]
Pasaje Granizo con Ramayana
[(-33.0624463, -71.3845345)]
Pasaje Las Cruces con Alcalde Miguel Gandulfo G.
[(-33.0588079, -71.3933562)]
Bilbao con Paul Harris
[(-33.0645217, -71.3618842)]
Bilbao con Pasaje Bilbao
[(-33.0645399, -71.3627146)]
Bilbao con Ignacio Carrera Pinto
[(-33.0646437, -71.3643767)]
Bilbao con Marcelino Champagnat
[(-33.0645495, -71.3631444)]
Pasaje Eugenio Grandi con La Serena
[(-33.0543387, -71.3812692)]
Pasaje Eugenio Grandi con Pasaje C
[(-33.0540322, -71.3818995)]
Pasaje Eugenio Grandi con Pasaje B
[(-33.0528698, -71.3820593)]
Pasaje Eugenio Grandi con Márquez
[(-33.051378, -71.3822581)]
Pasaje Eugenio Grandi con El Bosque
[(-33.0536011, -71.3819607)]
Pasaje Eugenio Grandi con Araya
[(-33.0524349, -71.3821201)]
Pasaje Sandy Zañartu con Pasaje Alfonso Alcalde
[(-33.062943, -71.3959291)]
Pasaje Sandy 

In [15]:
df2 = pd.DataFrame({'Calles': calles, 'Intersecciones': inters2, 'X': X, 'Y': Y})
df2.to_csv('D:/cbva/backend/inters.csv', sep='#', index=False, encoding='utf-8')